# Municipal Population Dataset (1996–2025): Cleaning and Standardisation
## *Preprocessing workflow for INE municipal time-series data*

This notebook downloads, parses, and cleans the INE Padrón Municipal histórico
(table 29005) programmatically via the INE JSON API, replacing the previous
manual CSV download workflow.

> **Update — April 2026:** Data source migrated from local CSV file to INE JSON API
> (`servicios.ine.es/wstempus/js/ES/DATOS_TABLA/29005?tip=AM`). Series extended to 2025.
> Encoding issues (latin1 corruption) eliminated at source.
> The API already excludes 2 ghost codes (12066, 17122) present in the manual download,
> delivering **8,136 raw → 8,132 clean** municipalities.

In [ ]:
"""
Notebook: 01_data_cleaning_padron_historico.ipynb
Author:   Juan Zotes
Created:  Dec 2025 (exploratory phase)
Last updated: 2026-04

Purpose:
    Download and clean historical municipal census data (Padrón Municipal)
    for Spain (1996–2025), ensuring consistency across years and municipalities.

Input:
    INE JSON API — table 29005 (Padrón Municipal histórico por municipio y sexo)
    URL: https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/29005?tip=AM
    tip=AM parameter returns data + metadata (including municipality codes)

Output:
    01_padron_clean_1996_2025.csv  — cleaned, harmonized demographic dataset

Spatial unit:
    Municipality (INE 5-digit code, Mun_Code)

Notes:
    - Focuses exclusively on data cleaning and preprocessing
    - Analytical steps are handled in subsequent notebooks (p0, p1a, p1b...)
    - Raw API data: 8,136 municipality codes (ghost codes already excluded by API)
    - Clean data: 8,132 municipalities (4 old merger codes aggregated and removed)
    - 1997 not present in API response (no Padron conducted that year)
    - Municipality count varies by year in raw data (newly created municipalities
      only appear from their creation year onwards)
"""

## 1. Load required libraries and configure environment

In [ ]:
# Standard library
from pathlib import Path
import time
import warnings

# Third-party libraries
import pandas as pd
import requests

warnings.filterwarnings('ignore')

In [ ]:
# Output directory for processed data
PROCESSED_DIR = Path(
    r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain\data\demography\processed"
)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Processed dir: {PROCESSED_DIR}")

## 2. Download Padrón Municipal histórico via INE JSON API

### API structure

The INE JSON API (`tip=AM`) returns a list of records, one per municipality × sex combination.
Each record contains:
- **MetaData**: municipality code (`Codigo`) and name, sex category
- **Data**: list of `{Anyo, Valor}` pairs — one entry per census year

The download function flattens this nested structure into a standard tabular DataFrame.

**API endpoint:**
```
https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/29005?tip=AM
```

**`tip=AM` is required** — without it, the API response omits `MetaData` and
municipality codes are not available. The parameter is documented in the INE API reference.

**Why JSON API instead of CSV?**
- No encoding issues (UTF-8 natively — eliminates `latin1` corruption from manual downloads)
- Always returns the most recent data (2025 included automatically)
- Fully reproducible: no manual download step required
- Consistent with the API-first approach used in notebook 00

**Note on municipality count by year:** The API delivers only the years in which each
municipality existed. Newly created municipalities appear only from their creation year
onwards. This means the raw count per year grows from ~8,096 (1996) to 8,132 (2024–2025),
which is the correct expected behaviour.

In [ ]:
INE_API_URL = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/29005?tip=AM"


def download_padron_historico(
    url: str = INE_API_URL,
    max_retries: int = 3,
    retry_delay: float = 10.0,
) -> pd.DataFrame:
    """
    Download and parse the INE Padron Municipal historico (table 29005).

    Uses tip=AM parameter to include metadata (municipality codes) in response.
    Without tip=AM the API omits MetaData and municipality codes are unavailable.

    The API already excludes ghost codes (12066 Gatova, 17122 Palmerola)
    that were present in the manual CSV download. Raw output is 8,136 municipalities.

    Parameters
    ----------
    url : str
        INE API endpoint with tip=AM parameter.
    max_retries : int
        Retry attempts on transient HTTP errors.
    retry_delay : float
        Seconds between retries (INE API can be slow for large tables).

    Returns
    -------
    pd.DataFrame
        Columns: Mun_Code (str, 5 digits), Mun (str), Cat (str),
                 Year (int), Pop (float)
    """
    print(f"Downloading Padron Municipal historico from INE API...")
    print(f"  URL: {url}")
    print(f"  Note: tip=AM includes metadata with municipality codes")
    print(f"  Warning: large request (~150MB), may take 1-2 minutes...")
    print()

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, timeout=180)
            response.raise_for_status()
            break
        except requests.RequestException as e:
            if attempt < max_retries:
                print(f"  Warning: Attempt {attempt} failed: {e}. Retrying in {retry_delay}s...")
                time.sleep(retry_delay)
            else:
                raise

    raw_data = response.json()
    print(f"  API response received: {len(raw_data)} records")

    # Parse nested JSON into flat rows
    # Each record = one municipality x sex combination
    # MetaData contains Mun_Code and Cat; Data contains one entry per year
    rows = []
    for record in raw_data:
        mun_code = None
        mun_name = None
        cat      = None

        for meta in record.get("MetaData", []):
            variable = meta.get("T3_Variable", "")
            if variable == "Municipios":
                mun_code = str(meta["Codigo"]).zfill(5)
                mun_name = meta["Nombre"]
            elif variable == "Sexo":
                cat = meta["Nombre"]   # 'Total', 'Hombres', 'Mujeres'

        if mun_code is None or cat is None:
            continue

        for entry in record.get("Data", []):
            year  = entry.get("Anyo")
            valor = entry.get("Valor")
            if year is not None:
                rows.append({
                    "Mun_Code": mun_code,
                    "Mun":      mun_name,
                    "Cat":      cat,
                    "Year":     int(year),
                    "Pop":      float(valor) if valor is not None else None,
                })

    df = pd.DataFrame(rows)

    print(f"  Parsed: {len(df):,} rows")
    print(f"  Unique municipalities : {df['Mun_Code'].nunique()}")
    print(f"  Year range            : {df['Year'].min()} - {df['Year'].max()}")
    print(f"  Sex categories        : {sorted(df['Cat'].unique().tolist())}")

    return df


# Execute download
padron_hist = download_padron_historico()

## 3. Preview raw data

In [ ]:
print("=== HEAD ===")
print(padron_hist.head())
print("\n=== TAIL ===")
print(padron_hist.tail())
print(f"\nShape: {padron_hist.shape}")

## 4. Set data types

In [ ]:
# Enforce correct types
# Mun_Code stays as string to preserve leading zeros (e.g. '01001')
padron_hist = padron_hist.astype({
    "Mun_Code": "string",
    "Mun":      "string",
    "Cat":      "string",
    "Year":     "int64",
    # Pop remains float64 to accommodate NaN values
})

# Verify leading zeros are preserved
sample = padron_hist["Mun_Code"].head(5).tolist()
assert all(len(c) == 5 for c in sample), "ERROR: Mun_Code lost leading zeros!"
print(f"Mun_Code format correct (5 digits). Sample: {sample}")
print(f"\nDtypes:\n{padron_hist.dtypes}")

## 5. Raw data checks

The API does not return 1997 (no Padrón conducted that year) — confirmed below.

Municipality count per year is not constant: newly created municipalities (codes `xx9xx`,
created by segregation) appear only from their creation year onwards. The count grows
from ~8,096 in 1996 to 8,132 in 2024–2025. This is expected and correct — the cleaning
step (Section 6) consolidates the series to 8,132 for all years.

In [ ]:
# Confirm 1997 is absent
assert 1997 not in padron_hist['Year'].values, "1997 unexpectedly present"
print(f"Year 1997 not present (no Padron conducted that year)")
print(f"Year range: {padron_hist['Year'].min()} - {padron_hist['Year'].max()}")

# Municipality count per year — expected to grow towards 8,132
mun_count_by_year = padron_hist.groupby("Year")["Mun_Code"].nunique()
print(f"\nMunicipality count per year (raw, before cleaning):")
print(mun_count_by_year.to_string())

# Confirm final years have reached 8,132
assert mun_count_by_year[2025] == 8132, (
    f"Expected 8,132 in 2025, got {mun_count_by_year[2025]}"
)
print(f"\n2024-2025: 8,132 municipalities (matches INE official count)")

# Non-negative population check
assert (padron_hist["Pop"].dropna() >= 0).all(), "Non-null population values must be >= 0"
print(f"All non-null population values are non-negative")
print(f"NaN values in Pop: {padron_hist['Pop'].isna().sum():,}")

## 6. Data cleaning: historical mergers

The INE API already excludes the 2 ghost codes (12066 Gatova, 17122 Palmerola)
that were present in the manual CSV download. The raw API count is therefore
**8,136** instead of the 8,138 seen previously.

The remaining difference of 4 codes comprises municipalities that merged during
the 1996–2025 time series:

| Old Code | Old Name | New Code | New Name | Merge Year |
|----------|----------|----------|----------|------------|
| 15026 | Cesuras | 15902 | Oza-Cesuras | 2013 |
| 15063 | Oza dos Ríos | 15902 | Oza-Cesuras | 2013 |
| 36011 | Cerdedo | 36902 | Cerdedo-Cotobade | 2016 |
| 36012 | Cotobade | 36902 | Cerdedo-Cotobade | 2016 |

The new codes (`15902`, `36902`) already exist in the Padrón from their merge year onwards.
Old codes are summed into the new code for all pre-merge years, then removed.
This produces a continuous time series at the current administrative boundary.

**Result: 8,136 − 4 (old merger codes) = 8,132** ✓

In [ ]:
# ==============================================================================
# CLEAN PADRON: AGGREGATE HISTORICAL MERGERS
# ==============================================================================
# Ghost codes (12066, 17122) are already excluded by the INE API.
# Only historical mergers require handling here.

# Historical mergers: new_code -> {new_name, merge_year, old_codes: {code: name}}
HISTORICAL_MERGERS = {
    '15902': {
        'new_name':   'Oza-Cesuras',
        'merge_year': 2013,
        'old_codes':  {'15026': 'Cesuras', '15063': 'Oza dos Rios'}
    },
    '36902': {
        'new_name':   'Cerdedo-Cotobade',
        'merge_year': 2016,
        'old_codes':  {'36011': 'Cerdedo', '36012': 'Cotobade'}
    }
}

print("\n" + "="*70)
print("CLEANING PADRON: AGGREGATING HISTORICAL MERGERS")
print("="*70)

count_before = padron_hist['Mun_Code'].nunique()
print(f"\nMunicipalities before cleaning : {count_before}")
print("(Ghost codes 12066, 17122 already excluded by INE API)")

print(f"\n" + "-"*70)
print("AGGREGATING HISTORICAL MERGERS")
print("-"*70)
print("Old municipality data is summed into the existing new municipality code.\n")

for new_code, info in HISTORICAL_MERGERS.items():
    old_codes = list(info['old_codes'].keys())

    print(f"  {' + '.join([f'{c} {n}' for c, n in info['old_codes'].items()])}")
    print(f"  -> {new_code} {info['new_name']} (merged {info['merge_year']})")

    for year in padron_hist['Year'].unique():
        for cat in padron_hist['Cat'].unique():
            old_pop = padron_hist[
                (padron_hist['Mun_Code'].isin(old_codes)) &
                (padron_hist['Year'] == year) &
                (padron_hist['Cat'] == cat)
            ]['Pop'].sum()

            if pd.notna(old_pop) and old_pop > 0:
                mask = (
                    (padron_hist['Mun_Code'] == new_code) &
                    (padron_hist['Year'] == year) &
                    (padron_hist['Cat'] == cat)
                )

                if mask.any():
                    current_pop = padron_hist.loc[mask, 'Pop'].iloc[0]
                    if pd.isna(current_pop):
                        padron_hist.loc[mask, 'Pop'] = old_pop
                    else:
                        padron_hist.loc[mask, 'Pop'] = current_pop + old_pop
                else:
                    new_row = pd.DataFrame({
                        'Mun_Code': [new_code],
                        'Mun':      [info['new_name']],
                        'Cat':      [cat],
                        'Year':     [year],
                        'Pop':      [old_pop]
                    })
                    padron_hist = pd.concat(
                        [padron_hist, new_row], ignore_index=True
                    )

    padron_hist = padron_hist[
        ~padron_hist['Mun_Code'].isin(old_codes)
    ].copy()

    # Verification sample
    sample = padron_hist[
        (padron_hist['Mun_Code'] == new_code) &
        (padron_hist['Cat'] == 'Total') &
        (padron_hist['Year'].isin([1996, 2010, 2020, 2025]))
    ].sort_values('Year')
    print(f"  Sample (Total) after aggregation:")
    for _, row in sample.iterrows():
        pop_str = f"{row['Pop']:.0f}" if pd.notna(row['Pop']) else "NaN"
        print(f"    {int(row['Year'])}: {pop_str}")
    print()

# Validation
print("-"*70)
print("VALIDATION")
print("-"*70)

count_final = padron_hist['Mun_Code'].nunique()
INE_OFFICIAL_COUNT = 8132

print(f"\n  Before cleaning              : {count_before}")
print(f"  Ghost codes (API excluded)   : -0 (already absent)")
print(f"  Old merger codes removed     : -4 (15026, 15063, 36011, 36012)")
print(f"  Final                        : {count_final}")

assert count_final == INE_OFFICIAL_COUNT, (
    f"Expected {INE_OFFICIAL_COUNT}, got {count_final}"
)
print(f"\nFinal count matches INE official figure ({INE_OFFICIAL_COUNT})")

# Sort for consistent output
padron_hist = padron_hist.sort_values(
    ['Mun_Code', 'Year', 'Cat']
).reset_index(drop=True)

## 7. Final dataset summary

In [ ]:
print("\n" + "="*70)
print("FINAL DATASET SUMMARY")
print("="*70)

print(f"\n  Rows          : {len(padron_hist):,}")
print(f"  Municipalities: {padron_hist['Mun_Code'].nunique():,}")
print(f"  Years         : {padron_hist['Year'].min()} - {padron_hist['Year'].max()}")
print(f"  Sex categories: {sorted(padron_hist['Cat'].unique().tolist())}")
print(f"  NaN in Pop    : {padron_hist['Pop'].isna().sum():,}")

print(f"\n  Columns:")
for col, dtype in padron_hist.dtypes.items():
    print(f"    {col:<12} {dtype}")

print(f"\n  Sample:")
print(padron_hist.head(10).to_string(index=False))

# Final assertions
assert padron_hist['Mun_Code'].nunique() == 8132
assert (padron_hist['Mun_Code'].str.len() == 5).all()
assert 1997 not in padron_hist['Year'].values
assert padron_hist['Year'].max() == 2025

print("\nAll assertions passed. Dataset ready for export.")
print("="*70)

## 8. Export clean dataset

In [ ]:
# ==============================================================================
# EXPORT TO CSV
# CSV is the canonical intermediate format for all downstream notebooks.
# UTF-8 with BOM (utf-8-sig) ensures compatibility with Excel and QGIS.
# ==============================================================================

output_file = PROCESSED_DIR / "01_padron_clean_1996_2025.csv"

padron_hist.to_csv(
    output_file,
    sep      = ",",
    index    = False,
    encoding = "utf-8-sig",
)

print("\n" + "="*70)
print("EXPORT COMPLETE")
print("="*70)
print(f"\n  File     : {output_file}")
print(f"  Size     : {output_file.stat().st_size / 1024 / 1024:.1f} MB")
print(f"  Rows     : {len(padron_hist):,}")
print(f"  Encoding : UTF-8 with BOM (utf-8-sig)")
print(f"  Sep      : comma")
print("\n  Columns:")
for col in padron_hist.columns:
    print(f"    {col}")

## Conclusion

The INE Padrón Municipal histórico (1996–2025) has been downloaded programmatically
via the INE JSON API (`tip=AM`), parsed from its nested structure into a flat tabular
format, and cleaned through the following steps:

- Ghost codes (12066 Gatova, 17122 Palmerola) already excluded by the API
- 4 old merger codes aggregated retrospectively into current codes:
  Oza-Cesuras (2013) and Cerdedo-Cotobade (2016)
- Final dataset: **8,132 municipalities × 28 years × 3 sex categories**

The dataset is exported as `01_padron_clean_1996_2025.csv` in UTF-8 encoding,
replacing the previous `latin1` encoding from the manual CSV workflow.

The CSV is the canonical intermediate format for all downstream notebooks.
It feeds into p0 (Goerlich integration), p1a (annual population change),
p1b (migratory balance), p1c (comparison), and will be integrated into
the geodatabase analítica municipal (Paper 3, Phase A).

### Next steps

The next notebook (p0) integrates this cleaned Padrón data with the Goerlich 2016
typology, resolves boundary changes (segregations and fusions), and produces the
consolidated `p0_padron_goerlich_1996_2025.csv` input for all analytical notebooks.